# CSCI 5922 — Experiments
**Player Number-Guided AI Highlight Video Curator**  
Mateusz Muszynski & Colin Wallace  

This notebook reproduces:
- **Experiment 1** — Jersey number recognition accuracy (top-1 digit-pair, per-digit, orientation filter recall)
- **Experiment 2** — Full pipeline highlight retrieval quality (precision / recall / F1, identity accuracy)

In [ ]:
import sys
sys.path.insert(0, '..')  # project root

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch
import cv2
from tqdm.notebook import tqdm

from src.utils import load_config, get_device
from src.detector import PlayerDetector
from src.jersey_reader import JerseyReader
from src.tracker import PlayerTracker
from src.scorer import HighlightScorer

cfg    = load_config('../config.yaml')
device = get_device()
print(f'Device: {device}')

---
## Experiment 1 — Jersey Number Recognition Accuracy

**Goal:** Measure how reliably Stage 2 reads the correct jersey number  
**Metrics:**  
- Top-1 digit-pair accuracy (primary)  
- Per-digit accuracy (tens place vs units place)  
- Orientation filter false-reject rate

In [ ]:
# ── Load the jersey CNN ──────────────────────────────────────────────────────
jr_cfg = cfg['jersey_reader']
reader = JerseyReader(
    model_path='../'+jr_cfg['model_path'],
    num_classes=jr_cfg['num_classes'],
    input_size=tuple(jr_cfg['input_size']),
    conf_threshold=jr_cfg['conf_threshold'],
    orientation_enabled=jr_cfg['orientation_filter']['enabled'],
    facing_threshold=jr_cfg['orientation_filter']['facing_threshold'],
    device=device,
)
print('JerseyReader loaded')

In [ ]:
# ── Evaluate on the SoccerNet jersey test split ───────────────────────────────
from pathlib import Path
from PIL import Image
import numpy as np

test_dir = Path('../data/soccernet_jersey')

results = []  # list of (gt_label, pred_label, confidence, was_rejected)

for label_dir in sorted(test_dir.iterdir()):
    if not label_dir.is_dir():
        continue
    try:
        gt_label = int(label_dir.name)
    except ValueError:
        continue
    
    for img_path in list(label_dir.glob('*.jpg'))[:50]:  # cap per-class for speed
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            continue
        pred, conf = reader.read(img_bgr)
        rejected = (pred is None)
        results.append({
            'gt':       gt_label,
            'pred':     pred if pred is not None else -1,
            'conf':     conf,
            'rejected': rejected,
        })

import pandas as pd
df = pd.DataFrame(results)
print(f'Evaluated {len(df)} crops')
df.head(10)

In [ ]:
# ── Metrics ──────────────────────────────────────────────────────────────────
accepted   = df[~df['rejected']]
top1_acc   = (accepted['gt'] == accepted['pred']).mean()
reject_rate = df['rejected'].mean()

# Per-digit accuracy (tens vs units)
accepted_copy = accepted.copy()
accepted_copy['gt_tens']   = accepted_copy['gt']   // 10
accepted_copy['pred_tens'] = accepted_copy['pred'] // 10
accepted_copy['gt_units']  = accepted_copy['gt']   % 10
accepted_copy['pred_units']= accepted_copy['pred'] % 10
tens_acc  = (accepted_copy['gt_tens']  == accepted_copy['pred_tens']).mean()
units_acc = (accepted_copy['gt_units'] == accepted_copy['pred_units']).mean()

print(f'Top-1 digit-pair accuracy : {top1_acc:.4f}')
print(f'Tens-place accuracy        : {tens_acc:.4f}')
print(f'Units-place accuracy       : {units_acc:.4f}')
print(f'Orientation filter reject  : {reject_rate:.4f}  ({df["rejected"].sum()} / {len(df)} crops)')

In [ ]:
# ── Confusion matrix (top-20 most common jersey numbers) ─────────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

top20 = accepted['gt'].value_counts().head(20).index.tolist()
subset = accepted[accepted['gt'].isin(top20)]

cm = confusion_matrix(subset['gt'], subset['pred'], labels=top20)
fig, ax = plt.subplots(figsize=(12, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=top20)
disp.plot(ax=ax, colorbar=False, xticks_rotation=45)
ax.set_title('Jersey Number Confusion Matrix (top-20 classes)')
plt.tight_layout()
plt.savefig('../outputs/exp1_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ── Confidence histogram ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

correct   = accepted[accepted['gt'] == accepted['pred']]['conf']
incorrect = accepted[accepted['gt'] != accepted['pred']]['conf']

axes[0].hist(correct,   bins=20, color='steelblue', alpha=0.7, label='Correct')
axes[0].hist(incorrect, bins=20, color='salmon',    alpha=0.7, label='Incorrect')
axes[0].set_xlabel('Softmax confidence')
axes[0].set_ylabel('Count')
axes[0].set_title('Confidence Distribution')
axes[0].legend()

# Per-class accuracy
per_class = accepted.groupby('gt').apply(
    lambda g: (g['gt'] == g['pred']).mean()
).reset_index()
per_class.columns = ['jersey_number', 'accuracy']
axes[1].bar(per_class['jersey_number'].astype(str), per_class['accuracy'], color='steelblue')
axes[1].set_xlabel('Jersey number')
axes[1].set_ylabel('Top-1 accuracy')
axes[1].set_title('Per-Class Accuracy')
axes[1].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.savefig('../outputs/exp1_confidence_hist.png', dpi=150)
plt.show()

---
## Experiment 2 — Full Pipeline Highlight Retrieval Quality

**Goal:** End-to-end evaluation — does the pipeline produce a useful player-specific reel?  
**Metrics:**  
- Precision / Recall / F1 at clip level (3-second tolerance window)  
- Identity accuracy (% of predicted clips that actually contain the target player)

In [ ]:
# ── Configure the test video and ground-truth annotations ────────────────────
# Replace these paths with your test video and ground-truth annotation file.

TEST_VIDEO      = '../data/test_match.mp4'
TARGET_JERSEY   = 10
ANNOT_FILE      = '../data/test_annotations.json'

# Ground-truth annotation format (JSON):
# [
#   {"start_frame": 450, "end_frame": 540, "jersey": 10},
#   ...
# ]

import json
from pathlib import Path

if not Path(TEST_VIDEO).exists():
    print(f'WARNING: test video not found at {TEST_VIDEO}')
    print('Set TEST_VIDEO to a real match video to run Experiment 2.')
else:
    print(f'Test video: {TEST_VIDEO}')
    print(f'Target jersey: #{TARGET_JERSEY}')

In [ ]:
# ── Run the full pipeline ─────────────────────────────────────────────────────
from main import run_pipeline

# Only run if test video exists
if Path(TEST_VIDEO).exists():
    result_path = run_pipeline(
        video_path=TEST_VIDEO,
        jersey_number=TARGET_JERSEY,
        output_path=f'../outputs/exp2_jersey{TARGET_JERSEY}.mp4',
        config_path='../config.yaml',
        device=device,
    )
    print(f'Highlight reel: {result_path}')
else:
    print('Skipping — set TEST_VIDEO above.')

In [ ]:
# ── Compute precision / recall / F1 ──────────────────────────────────────────
from src.utils import compute_clip_f1

if Path(TEST_VIDEO).exists() and Path(ANNOT_FILE).exists():
    annots = json.loads(Path(ANNOT_FILE).read_text())
    gt_clips = [
        (a['start_frame'], a['end_frame'])
        for a in annots
        if a.get('jersey') == TARGET_JERSEY
    ]

    # Load predicted clips from scorer (re-run or cache from pipeline run)
    # For demo: use gt_clips as predicted (replace with real pipeline output)
    pred_clips = gt_clips  # placeholder

    precision, recall, f1 = compute_clip_f1(
        predicted_frames=pred_clips,
        ground_truth_frames=gt_clips,
        tolerance_frames=90,   # 3 s at 30 fps
    )

    print(f'Precision : {precision:.4f}')
    print(f'Recall    : {recall:.4f}')
    print(f'F1        : {f1:.4f}')
    print(f'GT clips  : {len(gt_clips)}')
    print(f'Pred clips: {len(pred_clips)}')
else:
    print('Skipping metric computation — test video / annotations not found.')

In [ ]:
# ── PR curve (vary highlight threshold) ──────────────────────────────────────
# Demonstrates sensitivity of F1 to the scorer threshold.

thresholds  = np.linspace(0.3, 0.9, 13)
# Placeholder scores — replace with scorer.score_video() output per clip
mock_scores = np.random.rand(50)
mock_labels = (mock_scores > 0.55).astype(int)  # synthetic labels

precisions, recalls, f1s = [], [], []
for thr in thresholds:
    preds  = mock_scores >= thr
    tp     = int((preds & mock_labels.astype(bool)).sum())
    fp     = int((preds & ~mock_labels.astype(bool)).sum())
    fn     = int((~preds & mock_labels.astype(bool)).sum())
    p      = tp / (tp + fp + 1e-9)
    r      = tp / (tp + fn + 1e-9)
    f      = 2 * p * r / (p + r + 1e-9)
    precisions.append(p); recalls.append(r); f1s.append(f)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(thresholds, precisions, label='Precision', marker='o')
axes[0].plot(thresholds, recalls,    label='Recall',    marker='s')
axes[0].plot(thresholds, f1s,        label='F1',        marker='^')
axes[0].set_xlabel('Highlight threshold')
axes[0].set_title('P/R/F1 vs Threshold')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(recalls, precisions, marker='o', color='purple')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].grid(True)

plt.tight_layout()
plt.savefig('../outputs/exp2_pr_curve.png', dpi=150)
plt.show()